In [1]:
import numpy as np
import pandas as pd
from google.colab import drive
import ast

In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Hackathons

In [3]:
hackathons = pd.read_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/2020-2025_Hackathons.csv')

In [4]:
hackathons.shape

(4309, 13)

In [5]:
hackathons.head(3)

,hackathon_id,hackathon_slug,hackathon_title,status,url,submission_period,themes,prize_amount,registrations_count,organization,winners_announced,projects_gallery,projects_count
0,16823,orange-inter-odc-makeathon,Orange Inter-ODC Makeathon 2022,ended,https://orange-inter-odc-makeathon.devpost.com/,"Dec 03 - 04, 2022","['IoT', 'Enterprise', 'Web']",6000,23,Orange Digital Center,True,https://orange-inter-odc-makeathon.devpost.com...,12
1,16348,code-kevudah-hackathon,Code Kevudah Hackathon,ended,https://code-kevudah-hackathon.devpost.com/,"Nov 20, 2022","['Beginner Friendly', 'Open Ended']",4,23,Code Kevudah,True,https://code-kevudah-hackathon.devpost.com/pro...,9
2,16685,taws,Datathon TAWS,ended,https://taws.devpost.com/,"Nov 11 - 12, 2022","['Beginner Friendly', 'Machine Learning/AI', '...",180,23,TAWS,True,https://taws.devpost.com/project-gallery,6


In [6]:
hackathons.duplicated().sum()

np.int64(0)

In [7]:
hackathons_clean = hackathons.drop(columns=['hackathon_title', 'status', 'url', 'submission_period', 'winners_announced', 'projects_gallery'])
# winners_announced -> True

In [8]:
hackathons_clean.head(3)

,hackathon_id,hackathon_slug,themes,prize_amount,registrations_count,organization,projects_count
0,16823,orange-inter-odc-makeathon,"['IoT', 'Enterprise', 'Web']",6000,23,Orange Digital Center,12
1,16348,code-kevudah-hackathon,"['Beginner Friendly', 'Open Ended']",4,23,Code Kevudah,9
2,16685,taws,"['Beginner Friendly', 'Machine Learning/AI', '...",180,23,TAWS,6


## Projects

In [9]:

projects = pd.read_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/2020-2025_Projects.csv')

In [10]:
projects.shape

(85415, 8)

In [11]:
projects.head(1)

,hackathon_id,hackathon_slug,project_title,project_description,project_link,participants_count,likes,comments
0,16823,orange-inter-odc-makeathon,Owl4 -INSAT Team1,This solution is capable of automating the ind...,https://devpost.com/software/owl4,4,0,0


In [12]:
projects.duplicated().sum()

np.int64(0)

### Add project slug to Projects Table

In [13]:
projects['project_slug'] = (
    projects['project_link'].str.extract(r'devpost\.com/software/([^/]+)')
)

In [14]:
projects[['project_link', 'project_slug']].head()

,project_link,project_slug
0,https://devpost.com/software/owl4,owl4
1,https://devpost.com/software/here-we-go-l0z7rv,here-we-go-l0z7rv
2,https://devpost.com/software/my-frum-calendar,my-frum-calendar
3,https://devpost.com/software/mother-s-helper,mother-s-helper
4,https://devpost.com/software/g5-data-warriors,g5-data-warriors


In [15]:
projects.duplicated(subset='project_slug').sum()

np.int64(3071)

In [16]:
projects_clean = projects.drop_duplicates(subset='project_slug')

In [17]:
projects_clean.shape

(82344, 9)

In [18]:
projects_clean = projects_clean.drop(columns=['project_title', 'project_link'])

In [19]:
projects_clean.head(1)

,hackathon_id,hackathon_slug,project_description,participants_count,likes,comments,project_slug
0,16823,orange-inter-odc-makeathon,This solution is capable of automating the ind...,4,0,0,owl4


In [20]:
projects_clean.rename(
    columns={
        'project_description': 'short_description'
    },
    inplace=True
)

## Teams

In [21]:
teams = pd.read_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/2020-2025_Teams.csv')

In [22]:
teams.shape

(78288, 7)

In [23]:
teams.head(5)

,hackathon_id,project_slug,members_count,is_winner,winners_description,project_tags,project_description
0,11288,sharkroom,3,True,Best Domain Name from GoDaddy R...,"['bootstrap', 'collab', 'gcp', 'html5', 'javas...","Inspiration In this COVID-19 pandemic, teachin..."
1,11288,merry-cacti,4,True,Team Submission Prize - $25 USD...,"['css3', 'html5', 'javascript', 'materialui', ...",Merry Cacti Inspiration ⚡ Mental health is a s...
2,11288,remedy-sync,3,True,"Top 20 teams\n , ...","['angular.js', 'css3', 'firebase', 'html5']",Team name: Tech Exploders Inspiration 🤩 When p...
3,14065,babel-fish-z2vdh6,4,True,The Best Use of Assembly AI API...,"['assemblyai', 'dart', 'deep-translator', 'flu...",Inspiration The Babel Fish from Hitchhiker's G...
4,14065,adventure-addict,4,True,Best Hack | Runner Up 1\n ...,"['canva', 'dart', 'firebase', 'flutter', 'java...",💡 Inspiration 💡 We love sharing and writing st...


In [24]:
teams.duplicated().sum()

np.int64(115)

In [25]:
teams_clean = teams.drop_duplicates()

In [26]:
teams_clean.duplicated(subset='project_slug').sum()

np.int64(33)

In [27]:
teams_clean = teams_clean.drop_duplicates(subset="project_slug")

In [28]:
teams_clean.shape

(78140, 7)

### Drop Projects that don't appear on Teams Table

In [29]:
remaining_projects = set(projects_clean['project_slug']) - set(teams_clean['project_slug'])

In [30]:
len(remaining_projects)

4204

In [31]:
projects_clean = projects_clean[~projects_clean['project_slug'].isin(remaining_projects)]

In [32]:
projects_clean.shape

(78140, 7)

### Add hackathon_slug to Teams Table

In [33]:
teams_clean = teams_clean.merge(
    projects_clean[['hackathon_id', 'hackathon_slug']].drop_duplicates(),
    on='hackathon_id',
    how='left'
)

In [34]:
teams_clean[['hackathon_id', 'hackathon_slug']].head()

,hackathon_id,hackathon_slug
0,11288,sharkhacks
1,11288,sharkhacks
2,11288,sharkhacks
3,14065,hackumbc-fall-2021
4,14065,hackumbc-fall-2021


In [35]:
# teams_clean = teams_clean.drop(columns=['hackathon_id'])
# projects_clean = projects_clean.drop(columns=['hackathon_id'])
# hackathons_clean = hackathons_clean.drop(columns=['hackathon_id'])

In [36]:
print(teams_clean.shape)
print(projects_clean.shape)

(78140, 8)
(78140, 7)


## Team Members

In [37]:
team_members = pd.read_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/2020-2025_Team_Members.csv')

In [38]:
team_members.shape

(294843, 4)

In [39]:
team_members.head(5)

,project_slug,member_username,member_url,member_description
0,sharkroom,Harshita248,https://devpost.com/Harshita248,Worked a bit on frontend
1,sharkroom,Neilblaze,https://devpost.com/Neilblaze,NaN
2,sharkroom,sandipndev,https://devpost.com/sandipndev,NaN
3,merry-cacti,apurva-sharma668,https://devpost.com/apurva-sharma668,"I worked on the NLP Model, UI/UX design and lo..."
4,merry-cacti,cryptus-neoxys,https://devpost.com/cryptus-neoxys,I worked on the UI/UX design and Frontend Deve...


In [40]:
team_members.duplicated().sum()

np.int64(539)

In [41]:
team_members_clean = team_members.drop_duplicates()

## Members

In [42]:
members = pd.read_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/2020-2025_Members.csv')

In [43]:
members.shape

(243108, 15)

In [44]:
members.head(2)

,username,name,bio,location,website,github,hard_skills,hackathons_interests,project_count,hackathons_count,achievements_count,followers_count,following_count,likes_count,winnings_count
0,juniortaeza,Heginio Jr Taeza,Computer Science at the University of Southern...,"California, United States",NaN,https://github.com/juniortaeza,"['java', 'swift', 'unity', 'python', 'c++']","['AR/VR', 'Communication', 'Gaming', 'Health',...",3,3,5,1,1,4,2
1,micheledimuccio1,Michele Di Muccio,NaN,NaN,NaN,NaN,"['python', 'arduino', 'c++', 'dashboard', 'fig...","['AR/VR', 'Gaming', 'Health', 'IoT', 'Machine ...",1,3,5,0,0,1,1


In [45]:
members.duplicated().sum()

np.int64(27121)

In [46]:
members_clean = members.drop_duplicates()

In [47]:
members_clean.shape

(215987, 15)

In [48]:
len(members_clean[members_clean['username'] == 'Username'])

1

In [49]:
members_clean = members_clean.drop(members_clean[members_clean['username'] == 'Username'].index)

In [50]:
members_clean.shape

(215986, 15)

In [51]:
members_clean.duplicated(subset="username").sum()

np.int64(49)

In [52]:
members_clean = members_clean.drop_duplicates(subset="username")

In [53]:
members_clean.shape

(215937, 15)

In [54]:
members_clean.head(2)

,username,name,bio,location,website,github,hard_skills,hackathons_interests,project_count,hackathons_count,achievements_count,followers_count,following_count,likes_count,winnings_count
0,juniortaeza,Heginio Jr Taeza,Computer Science at the University of Southern...,"California, United States",NaN,https://github.com/juniortaeza,"['java', 'swift', 'unity', 'python', 'c++']","['AR/VR', 'Communication', 'Gaming', 'Health',...",3,3,5,1,1,4,2
1,micheledimuccio1,Michele Di Muccio,NaN,NaN,NaN,NaN,"['python', 'arduino', 'c++', 'dashboard', 'fig...","['AR/VR', 'Gaming', 'Health', 'IoT', 'Machine ...",1,3,5,0,0,1,1


In [55]:
members_clean=members_clean.dropna(subset=['username'])

In [56]:
members_clean = members_clean.drop(columns=['name', "location", 'website', 'github'])

## Add missing member_usernames to Team Member Table

### Fix NaN member_usernames

In [57]:
empty_usernames = team_members_clean[team_members_clean['member_username'].isna()]

In [58]:
empty_usernames.shape

(53, 4)

In [59]:
empty_usernames

,project_slug,member_username,member_url,member_description
1656,basket_analysis,NaN,https://www.basketball-reference.com/,I had worked on the back-end statistical and a...
2019,petme,NaN,https://akshitagupta15june.github.io/PetMe/,"Our project is PetMe, a platform that allows a..."
3255,insight-iqktcb,NaN,https://insitee.github.io/,"I created logos, helped with the machine learn..."
3941,operation-guesser,NaN,https://www.idonotwantdumbkids.com/,PLANNING:\nI came up with the idea to expand o...
4379,genesis-b3m6vf,NaN,http://genesis2.us-east-2.elasticbeanstalk.com/,I wrote the back-end code using Python and SQL...
6025,depress-the-depression,NaN,http://depress-the-depression-with.us/,I worked on the Augmented reality part of the ...
7796,rainbow-restroom,NaN,http://computers-are-binary-but-not-people.tech/,Have a look at this website here http://comput...
7892,covidtivity,NaN,https://grace-ling.com/,"I worked on all the UI/UX Design, Visual Desig..."
7893,covidtivity,NaN,https://www.linkedin.com/in/harjina/,I dedicated my time for this team in devising ...
11441,flowerdex,NaN,https://www.linkedin.com/in/hyunwin/,4th year Computer Science\nhttps://www.linkedi...


In [60]:
# empty_usernames.to_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/empty_usernames.csv', index=False)

In [61]:
fixed_empty_users = pd.read_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/fixed_empty_usernames.csv')

In [62]:
sample = pd.read_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/fixed_empty_sample.csv')

In [63]:
fixed_empty_users.head(2)

,project_slug,member_username,member_description
0,ucity-gzal3x,zphill15,I worked on connecting unity to a node.js serv...
1,holesovice-testnet,YazzyYaz,Spun up the ETH1 Testnet with difficulty set t...


In [64]:
sample.head(2)

,project_slug,member_username,member_url,member_description
0,petme,akshitagupta15june,https://akshitagupta15june.github.io/PetMe/,"Our project is PetMe, a platform that allows a..."
1,insight-iqktcb,makensonnoel,https://insitee.github.io/,"I created logos, helped with the machine learn..."


In [65]:
sample = sample.drop(columns='member_url')

In [66]:
fixed_empty_users = pd.concat(
    [fixed_empty_users, sample],
    ignore_index=True
)

In [67]:
fixed_empty_users.shape

(45, 3)

In [68]:
fixed_empty_users = fixed_empty_users.rename(columns={
    'member_username': 'member_username_fixed'
})

In [69]:
keys = ['project_slug', 'member_description']

In [70]:
tmp = team_members_clean.merge(
    fixed_empty_users[keys + ['member_username_fixed']],
    on=keys,
    how='left'
)

In [71]:
tmp['member_username'] = tmp['member_username'].fillna(tmp['member_username_fixed'])

In [72]:
team_members_clean = tmp.drop(columns=['member_username_fixed'])

In [73]:
team_members_clean['member_username'].isna().sum()

np.int64(13)

### Fix Invalid member_usernames

In [74]:
invalid = team_members_clean[
    team_members_clean['member_url'].notna() &
    ~team_members_clean['member_url'].str.startswith('https://devpost.com/')
]

In [75]:
invalid.shape

(340, 4)

In [76]:
# invalid.to_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/invalid_usernames.csv', index=False)

In [77]:
fixed_invalid_users = pd.read_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/fixed_invalid_usernames.csv')

In [78]:
fixed_invalid_users.head(1)

,project_slug,member_username,member_description
0,shopnation,markanthonyantao,Hi! My name is Mark Antao! I worked on the Fro...


In [79]:
fixed_invalid_users = fixed_invalid_users.rename(columns={
    'member_username': 'member_username_fixed'
})

In [80]:
keys = ['project_slug', 'member_description']

In [81]:
tmp = team_members_clean.merge(
    fixed_invalid_users[keys + ['member_username_fixed']],
    on=keys,
    how='left'
)

In [82]:
tmp['member_username'] = tmp['member_username_fixed'].combine_first(tmp['member_username'])
team_members_clean = tmp.drop(columns=['member_username_fixed'])

In [83]:
# no. of records updated
tmp['member_username_fixed'].notna().sum()

np.int64(95)

In [84]:
team_members_clean[team_members_clean['member_url']=='https://youtu.be/as-0t_VE9jU']

,project_slug,member_username,member_url,member_description
111394,sandwich-voice-driven-development-workflow,as-0t_VE9jU,https://youtu.be/as-0t_VE9jU,https://youtu.be/as-0t_VE9jU


In [85]:
team_members_clean.loc[
    (team_members_clean['project_slug'] == 'sandwich-voice-driven-development-workflow') &
    (team_members_clean['member_username'] == 'as-0t_VE9jU'),
    'member_username'
] = 'dhruvmiyani'


In [86]:
team_members_clean[team_members_clean['member_url']=='https://youtu.be/as-0t_VE9jU']

,project_slug,member_username,member_url,member_description
111394,sandwich-voice-driven-development-workflow,dhruvmiyani,https://youtu.be/as-0t_VE9jU,https://youtu.be/as-0t_VE9jU


In [87]:
team_members_clean.duplicated(subset=['project_slug', 'member_username']).sum()

np.int64(15)

In [88]:
duplicates = team_members_clean[
    team_members_clean.duplicated(
        subset=['project_slug', 'member_username'],
        keep=False
    )
]

duplicates

,project_slug,member_username,member_url,member_description
2425,charitychimp,watch?v=grd-K33tOSM,https://www.youtube.com/watch?v=grd-K33tOSM,"I worked mainly on the frontend, making the ho..."
2426,charitychimp,watch?v=grd-K33tOSM,https://www.youtube.com/watch?v=grd-K33tOSM,"I fixed some bugs on the frontend, made UI imp..."
2427,charitychimp,watch?v=grd-K33tOSM,https://www.youtube.com/watch?v=grd-K33tOSM,I worked on connecting Deso with express and d...
2428,charitychimp,watch?v=grd-K33tOSM,https://www.youtube.com/watch?v=grd-K33tOSM,"I worked all around, but mostly, I helped with..."
7892,covidtivity,NaN,https://grace-ling.com/,"I worked on all the UI/UX Design, Visual Desig..."
7893,covidtivity,NaN,https://www.linkedin.com/in/harjina/,I dedicated my time for this team in devising ...
9314,superior-tech-website-with-echoar,adityat21,https://devpost.com/adityat21,"I came up with the idea, and helped contribute..."
9315,superior-tech-website-with-echoar,adityat21,https://devpost.com/adityat21,NaN
11441,flowerdex,NaN,https://www.linkedin.com/in/hyunwin/,4th year Computer Science\nhttps://www.linkedi...
11443,flowerdex,NaN,https://www.linkedin.com/in/sebastian-wueste-0...,4th year EE\nhttps://www.linkedin.com/in/sebas...


In [89]:
invalid_users = set(duplicates['member_username'])

In [90]:
len(invalid_users)

11

In [91]:
members_clean[members_clean['username'].isin(invalid_users)]

,username,bio,hard_skills,hackathons_interests,project_count,hackathons_count,achievements_count,followers_count,following_count,likes_count,winnings_count
10049,JaryJay,Helo,"['java', 'github', 'python', 'kotlin', 'typesc...","['Gaming', 'Machine Learning/AI', 'Beginner Fr...",8,11,10,9,7,16,3
16309,adityat21,Young coder aspiring to be ML and AI professional,"['python', 'sql', 'html5', 'c']","['Machine Learning/AI', 'AR/VR', 'Cybersecurit...",3,4,5,1,5,3,1
51694,rpsammons6,NaN,['apis'],['Beginner Friendly'],2,1,5,1,1,0,0
156261,w1_VbP81EWI,NaN,[],[],0,0,0,0,0,0,0
156262,-4ScQIiUOys,NaN,[],[],0,0,0,0,0,0,0
174189,connermcguire,NaN,['javascript'],"['Beginner Friendly', 'Databases']",2,1,5,1,0,0,0
211504,Gurpinder-Grewal,NaN,"['html5', 'css3', 'java', 'javascript', 'node....","['DevOps', 'Gaming', 'IoT', 'Lifehacks', 'Mach...",2,1,4,0,1,2,0
212682,williamvnguyen2,NaN,"['java', 'python', 'react', 'c', 'c++', 'javas...","['Blockchain', 'Cybersecurity', 'DevOps', 'Fin...",2,1,4,0,0,1,0


In [92]:
dropped=['w1_VbP81EWI', '-4ScQIiUOys']
members_clean=members_clean[~members_clean['username'].isin(dropped)]

In [93]:
to_drop = ['notesdev', 'caravan-rkjwdg']

teams_clean = teams_clean[~teams_clean['project_slug'].isin(to_drop)]
team_members_clean = team_members_clean[~team_members_clean['project_slug'].isin(to_drop)]

In [94]:
team_members_clean.loc[
    (team_members_clean['project_slug'] == 'hermitrade') &
    (team_members_clean['member_description'] == 'We are.'),
    'member_username'
] = 'Dissonant101'

In [95]:
team_members_clean = team_members_clean.drop(columns=['member_url'])

## Remove Team with Private Users

In [96]:
team_members_clean.head(0)

,project_slug,member_username,member_description


In [97]:
# with project slug -> cs-voyager
projects_clean = projects_clean[projects_clean['project_slug'] != 'cs-voyager']
teams_clean = teams_clean[teams_clean['project_slug'] != 'cs-voyager']
team_members_clean = team_members_clean[team_members_clean['project_slug'] != 'cs-voyager']
members_clean = members_clean[members_clean['username'].isin(team_members_clean['member_username'])]

## clean teams from Not Found Members

In [98]:
exist_members = set(members_clean['username'])

In [99]:
len(exist_members)

215850

In [100]:
team_members_clean['is_member_found'] = team_members_clean['member_username'].isin(exist_members)

In [101]:
len(team_members_clean[team_members_clean['is_member_found'] == False])

669

In [102]:
team_members_clean['project_slug'].value_counts()

,count
project_slug,
consvid-la-tete-19,8
mapsmap-hackathon,8
vici,8
appian-scrum-board,8
ployify,8
...,...
nami-volo,2
wanitracker,2
try-our-404,2


In [103]:
projects_clean.duplicated(subset='project_slug').sum()

np.int64(0)

In [104]:
grouped_projects =(
    team_members_clean
    .groupby('project_slug')['is_member_found']
    .all().reset_index()
)

In [105]:
len(grouped_projects)

78132

In [106]:
len(grouped_projects[grouped_projects['is_member_found']==False])

607

In [107]:
valid_projects = grouped_projects[
    grouped_projects['is_member_found']
]['project_slug']

In [108]:
len(valid_projects)

77525

In [109]:
teams_clean = teams_clean[teams_clean['project_slug'].isin(valid_projects)]

In [110]:
teams_clean.shape

(77525, 8)

In [111]:
projects_clean = projects_clean[projects_clean['project_slug'].isin(valid_projects)]

In [112]:
projects_clean.shape

(77525, 7)

In [113]:
team_members_clean = team_members_clean[
    team_members_clean['project_slug'].isin(valid_projects)
]

In [114]:
team_members_clean.shape

(291860, 4)

In [115]:
team_members_clean = team_members_clean.drop(columns=['is_member_found'])

## Drop teams which have < 3 members

### clean team table

In [116]:
teams_clean.shape

(77525, 8)

In [117]:
teams_clean[teams_clean['members_count'] >= 3].shape

(77518, 8)

In [118]:
teams_clean = teams_clean[teams_clean['members_count'] >= 3]

In [119]:
teams_clean.shape

(77518, 8)

In [120]:
teams_clean.head(2)

,hackathon_id,project_slug,members_count,is_winner,winners_description,project_tags,project_description,hackathon_slug
0,11288,sharkroom,3,True,Best Domain Name from GoDaddy R...,"['bootstrap', 'collab', 'gcp', 'html5', 'javas...","Inspiration In this COVID-19 pandemic, teachin...",sharkhacks
1,11288,merry-cacti,4,True,Team Submission Prize - $25 USD...,"['css3', 'html5', 'javascript', 'materialui', ...",Merry Cacti Inspiration ⚡ Mental health is a s...,sharkhacks


### clean team_members table

In [121]:
team_members_clean.head(2)

,project_slug,member_username,member_description
0,sharkroom,Harshita248,Worked a bit on frontend
1,sharkroom,Neilblaze,NaN


In [122]:
team_members_clean.shape

(291860, 3)

In [123]:
team_members_clean = team_members_clean[
    team_members_clean['project_slug'].isin(teams_clean['project_slug'])
]

In [124]:
team_members_clean.shape

(291848, 3)

### clean members table

In [125]:
members_clean = members_clean[
    members_clean['username'].isin(team_members_clean['member_username'])
]

In [126]:
members_clean.shape

(214700, 11)

### clean projects table

In [127]:
projects_clean = projects_clean[projects_clean['project_slug'].isin(teams_clean['project_slug'])]

In [128]:
projects_clean.shape

(77518, 7)

## Convert all string lists to Lists

In [129]:
def to_list(col):
    if isinstance(col, list):
        return col
    if isinstance(col, str):
        try:
            return ast.literal_eval(col)
        except:
            return []
    return []

In [130]:
# members_clean['hard_skills'] = members_clean['hard_skills'].apply(to_list)
# members_clean['hackathons_interests'] = members_clean['hackathons_interests'].apply(to_list)

## Drop Projects according to Empty project_tags

In [131]:
# convert project_tags to list
teams_clean['project_tags'] = teams_clean['project_tags'].apply(to_list)

In [132]:
# check empty list in project_tags column
teams_clean['project_tags'].apply(lambda x: isinstance(x, list) and len(x) == 0).sum()

np.int64(515)

### clean teams table

In [133]:
teams_clean.shape

(77518, 8)

In [134]:
teams_clean = teams_clean[
    teams_clean['project_tags'].apply(lambda x: not (isinstance(x, list) and len(x) == 0))
]

In [135]:
teams_clean.shape

(77003, 8)

### clean team_members table

In [136]:
team_members_clean.shape

(291848, 3)

In [137]:
team_members_clean = team_members_clean[
    team_members_clean['project_slug'].isin(teams_clean['project_slug'])
]

In [138]:
team_members_clean.shape

(289685, 3)

### clean members table

In [139]:
members_clean.shape

(214700, 11)

In [140]:
members_clean = members_clean[
    members_clean['username'].isin(team_members_clean['member_username'])
]

In [141]:
members_clean.shape

(213005, 11)

### clean projects table

In [142]:
projects_clean.shape

(77518, 7)

In [143]:
projects_clean = projects_clean[
    projects_clean['project_slug'].isin(teams_clean['project_slug'])
]

In [144]:
projects_clean.shape

(77003, 7)

## Clean Hackathons Table

### Skipped
### Drop Hackathons with less than 10 projects

In [145]:
# hackathons_clean.shape

In [146]:
# len(hackathons_clean[hackathons_clean['projects_count'] < 10])

In [147]:
# hackathons_clean = hackathons_clean[hackathons_clean['projects_count'] >= 10]

In [148]:
# hackathons_clean.shape

### Drop Hackathons with unknown Organization

In [149]:
hackathons_clean.shape

(4309, 7)

In [150]:
hackathons_clean['organization'].isna().sum()

np.int64(693)

In [151]:
hackathons_clean = hackathons_clean[hackathons_clean['organization'].notna()]

In [152]:
hackathons_clean.shape

(3616, 7)

In [153]:
hackathons_clean = hackathons_clean.drop(columns=['organization'])

### Drop Hackathons with empty Themes

In [154]:
hackathons_clean['themes'] = hackathons_clean['themes'].apply(to_list)

In [155]:
hackathons_clean['themes'].apply(lambda x: isinstance(x, list) and len(x) == 0).sum()

np.int64(24)

In [156]:
hackathons_clean.shape

(3616, 6)

In [157]:
hackathons_clean = hackathons_clean[
    hackathons_clean['themes'].apply(lambda x: isinstance(x, list) and len(x) > 0)
]

In [158]:
hackathons_clean.shape

(3592, 6)

### Clean Projects Table

In [159]:
projects_clean.shape

(77003, 7)

In [160]:
projects_clean = projects_clean[
    projects_clean['hackathon_id'].isin(hackathons_clean['hackathon_id'])
]

In [161]:
projects_clean.shape

(63527, 7)

### Clean Teams Table

In [162]:
teams_clean.shape

(77003, 8)

In [163]:
teams_clean = teams_clean[
    teams_clean['project_slug'].isin(projects_clean['project_slug'])
]

In [164]:
teams_clean.shape

(63527, 8)

### Clean Team Members Table

In [165]:
team_members_clean.shape

(289685, 3)

In [166]:
team_members_clean = team_members_clean[
    team_members_clean['project_slug'].isin(teams_clean['project_slug'])
]

In [167]:
team_members_clean.shape

(239200, 3)

### Clean Members Table

In [168]:
members_clean.shape

(213005, 11)

In [169]:
members_clean = members_clean[
    members_clean['username'].isin(team_members_clean['member_username'])
]

In [170]:
members_clean.shape

(179103, 11)

### Clean Hackathons Table according to project_slug

In [171]:
hackathons_clean.shape

(3592, 6)

In [172]:
len(projects_clean['hackathon_id'].unique())

3396

In [173]:
hackathons_clean = hackathons_clean[
    hackathons_clean['hackathon_id'].isin(projects_clean['hackathon_id'])
]

In [174]:
hackathons_clean.shape

(3396, 6)

## Merge member_descriptions

In [175]:
team_members_clean.head(1)

,project_slug,member_username,member_description
10,babel-fish-z2vdh6,jhair1,I mainly worked on the transcript and translat...


In [176]:
team_members_clean['member_description'].isna().sum()

np.int64(181933)

In [177]:
# drop null and empty strings
tmp_team_members = team_members_clean.dropna(subset=['member_description'])
tmp_team_members = tmp_team_members[tmp_team_members['member_description'].str.strip() != ""]

In [178]:
member_desc_list = (
    tmp_team_members
    .groupby('member_username')['member_description']
    .apply(lambda x: list(x.unique()))
    .reset_index(name='Descriptions')
)

In [179]:
member_desc_list.head()

,member_username,Descriptions
0,-Kevin-,[I helped out with a mostly everything. I also...
1,-alam,[I've been deeply involved as a Smart Contract...
2,0-Yuuki-0,[I used SAP BTP to implement and deploy the ba...
3,00000ff000,[Domain Expert.]
4,0001programmer,[I contributed in providing and managing the c...


### Merge Descriptions into Members Table

In [180]:
members_clean = members_clean.merge(
   member_desc_list,
   left_on='username',
   right_on='member_username',
   how='left'
).drop(columns=['member_username'])

In [181]:
members_clean.head(2)

,username,bio,hard_skills,hackathons_interests,project_count,hackathons_count,achievements_count,followers_count,following_count,likes_count,winnings_count,Descriptions
0,dembiczakmatt,NaN,[],[],2,2,9,0,0,1,2,NaN
1,kkysharma2114,NaN,"['design', 'ui', 'ux', 'matlab', 'python', 'ht...","['Beginner Friendly', 'Communication', 'Design...",3,7,6,6,0,3,0,NaN


In [182]:
members_clean['Descriptions'].notna().sum()

np.int64(48049)

## Add main Project Description (from Projects table) to Teams table

In [183]:
projects_clean = projects_clean.rename(
    columns={
        'project_description': 'short_description'
    }
)

In [184]:
teams_clean = teams_clean.merge(
    projects_clean[['project_slug', 'short_description']],
    on='project_slug',
    how='left'
)

In [185]:
teams_clean[['project_slug', 'short_description']].head(3)

,project_slug,short_description
0,babel-fish-z2vdh6,Hitchhiker's Guide to the Galaxy is a classic ...
1,adventure-addict,Bit-sized adventures right at your fingertips.
2,status-bar,Looking at life's data at a glance. A table to...


In [186]:
teams_clean['short_description'].isna().sum()

np.int64(25)

In [187]:
teams_clean.shape

(63527, 9)

In [188]:
projects_clean['short_description'].isna().sum()

np.int64(25)

In [189]:
projects_clean.shape

(63527, 7)

## Check all Table

In [190]:
len(hackathons_clean[~hackathons_clean['hackathon_id'].isin(projects_clean['hackathon_id'])])

0

In [191]:
len(projects_clean[~projects_clean['hackathon_id'].isin(hackathons_clean['hackathon_id'])])

0

In [192]:
len(projects_clean[~projects_clean['project_slug'].isin(teams_clean['project_slug'])])

0

In [193]:
len(teams_clean[~teams_clean['project_slug'].isin(projects_clean['project_slug'])])

0

In [194]:
len(teams_clean[~teams_clean['project_slug'].isin(team_members_clean['project_slug'])])

0

In [195]:
len(team_members_clean[~team_members_clean['project_slug'].isin(teams_clean['project_slug'])])

0

In [196]:
len(team_members_clean[~team_members_clean['member_username'].isin(members_clean['username'])])

0

In [197]:
len(members_clean[~members_clean['username'].isin(team_members_clean['member_username'])])

0

## Save Cleanded Tables

In [198]:
print(hackathons_clean.shape)
print(projects_clean.shape)
print(teams_clean.shape)
print(team_members_clean.shape)
print(members_clean.shape)

(3396, 6)
(63527, 7)
(63527, 9)
(239200, 3)
(179103, 12)


In [199]:
len(teams_clean[teams_clean['is_winner'] == True])

15965

In [200]:
len(teams_clean[teams_clean['is_winner'] == False])

47562

In [201]:
hackathons_clean.to_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/Cleaned_Dataset/hackathons_clean.csv', index=False)
projects_clean.to_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/Cleaned_Dataset/projects_clean.csv', index=False)
teams_clean.to_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/Cleaned_Dataset/teams_clean.csv', index=False)
team_members_clean.to_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/Cleaned_Dataset/team_members_clean.csv', index=False)
members_clean.to_csv('/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/Cleaned_Dataset/members_clean.csv', index=False)